In [95]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import spacy
import re
import faiss
import contractions
from textblob import TextBlob
from nltk.tokenize import word_tokenize


### 1. Load the document (.txt)

In [96]:
data=open('data.txt').read()

### 2. Text Normalization


#### a. converting all the character into lowercase

In [97]:
data=data.lower()

#### b. removing Extra space

In [98]:
data=re.sub(r'\s{2,}','',data)

- removing numbers like 1.

In [99]:
# data=re.sub(r'')

#### c. Contraction

In [100]:
data=contractions.fix(data)  #check the return type before use

#### d. REmoving the punctuation and spl characters

In [101]:
data=re.sub(r'[^0-9a-zA-Z\s]','',data)

#### e. Textblob

- we use textblob to correct the sentence ie. spelling correction

In [ ]:
values=TextBlob(data).correct().raw_sentences
data=' '.join(values)


#### f. Spacy lamentization

In [103]:
nlp=spacy.load('en_core_web_sm')
tokens=nlp(data)
update_tokens=[token.lemma_ for token in tokens if not token.is_stop]
data=' '.join(update_tokens).strip()  # we use strip to extra spaces

#### g. chunking(converting the doc into chunks [doc-->chunk])

In [ ]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40
)
# Generate chunks from the lemmatized text
'''chunks = splitter.split_text(data)'''  # it return all chunks
chunks = list(set(splitter.split_text(data)))

In [105]:
print(len(chunks))

140


#### h. chunk embeddings (converting the chunks to vectors)

In [106]:
embedding_model=SentenceTransformer(
    model_name_or_path='sentence-transformers/all-miniLM-L6-V2'  #to convert the chunks into vector
)
chunk_embedding=embedding_model.encode(chunks).astype('float32')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [107]:
chunk_embedding.shape

(140, 384)

In [108]:
dimension=chunk_embedding.shape[1]  
dimension

384

In [109]:
faiss.normalize_L2(chunk_embedding)

In [110]:
index_faiss_db=faiss.IndexFlatIP(dimension)
index_faiss_db.add(chunk_embedding)

In [111]:
def rag_query(query,k=2):
    query_embedding=embedding_model.encode(query).astype('float32')
    query_embedding=query_embedding.reshape(1,-1)
    faiss.normalize_L2(query_embedding)
    print(query_embedding.shape)
    values=distance,index=index_faiss_db.search(query_embedding,k=k)
    print(values)

    R_chunks=[chunks[i] for i in index[0]] 
    R_str=' '.join(R_chunks)
    prompt= f''' 
                you're an helpful assistant 
                Assigned Task for you: Structure my output => {R_str}
                Note:
                1) Don't add extra contents just structure mentioned output.
                2) If there is mistake in output correct or else keep the original output
    
    '''


(1, 384)
(array([[0.57110596, 0.53118783]], dtype=float32), array([[2, 5]], dtype=int64))
machine learning important technology industry 
 healthcare finance education transportation retail entertainment cybersecurity 
 basic idea machine learning simple explainable ai technique attempt provide insight model predictionsthe machine learning lifecycle include stage 
 process begin understand business problem 
 step collect relevant datum


#### used hugging face

In [113]:
def r_search(query,k=3):
    query_embeddings = embedding_model.encode(query).astype('float32')
    query_embeddings = query_embeddings.reshape(1,-1)
    faiss.normalize_L2(query_embeddings)
    print(query_embeddings.shape)
    distance,index = index_faiss_db.search(query_embeddings,k=k)
    R_chunks = [chunks[i] for i in index[0]]
    R_str = ' '.join(R_chunks)
    return R_str
def g_text(r_search):
        import os
        import requests
    
        API_URL = "https://router.huggingface.co/v1/chat/completions"
    
        headers = {
            "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
        }
        def query(payload):
            response = requests.post(API_URL, headers=headers, json=payload)
            return response
        prompt = f'''
                    You're an helpful assistant
                    Assigned Task for you : Structure my output => {r_search}
                    Note : 
                    1) Don't add extra contents just structure mentioned output.
                    2) If there is mistake in output correct or else keep the original output
                    with structured result.
            '''
        response = query({
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "model": "deepseek-ai/DeepSeek-R1:novita"
        })
    
        return response
user_prompt = 'Explain Machine Learning ?'
user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)

r_response = r_search(user_prompt)
g_response = g_text(r_response)
print(g_response)

(1, 384)
<Response [200]>
